# iToF / Structured-Light Fusion Pipeline

Calibration and depth fusion for a combined indirect Time-of-Flight (iToF) and Structured Light (SL) approach.
Based on:
- **Microsoft-Paper** – Godbaz et al. 2025 (Microsoft): dot calibration (4.1), 1-D dot trails, consistency error, active brightness trail (4.2)
- **Agresti** – Agresti & Zanuttigh, ECCV 2018: maximum likelihood depth fusion (eq. 16/17)

The code lives in `dot_calibration.py`, `calibrate.py`, `approaches.py`, `sl_tilt_correction.py` and
`reconstruction.py`. This notebook only sets parameters and calls those modules.

**ToF preprocessing** (in `DotCalibration.load_tof_pcd`, so the calibration and the test scene see the same clouds; files on disk stay untouched):
- **SOR** (`dotCal.sor = (k, nσ)`): CloudCompare-style statistical outlier removal. A point is dropped when its mean distance to its k nearest neighbours exceeds mean + nσ·std over the cloud. Only the geometry is filtered; the active brightness is left as it is.
- **Light-path correction** (`LIGHT_PATH_CORRECTION`): the ToF is lit by the dot projector at B, not from the lens centre, so ranges are corrected via r = (4m² − |B|²) / (4m − 2 d̂·B). B is estimated iteratively during the calibration. **Off for v1.1 calibrations**, because their FPPN already contains the light path.

In [1]:
import matplotlib.pyplot as plt
from pathlib import Path

import approaches
import calibrate
import reconstruction
import sl_tilt_correction
from dot_calibration import DotCalibration, cache_tag, plot_sor_effect

CAMERA = "SchmersalReal_6.11"   # key of calibrate.CAMERAS
SOR = (6, 0.4)                     # (neighbours, nσ) or None
LIGHT_PATH_CORRECTION = False      # True only for clouds made with a pre-v1.1 ToF calibration

LOAD_CALIBRATION = False           # False: compute the calibration below
LOAD_CAL_NAME = f"{CAMERA}_paperlike{cache_tag(SOR, LIGHT_PATH_CORRECTION)}_06.11"  # None: newest matching CAMERA + tags
SAVE_CAL_NAME = f"{CAMERA}_paperlike{cache_tag(SOR, LIGHT_PATH_CORRECTION)}_06.11"

dotCal = DotCalibration()
dotCal.sor = SOR

## Calibration

With `LOAD_CALIBRATION = True`, `Calibrations/<LOAD_CAL_NAME>/calibration.json` is loaded, together with the light origin it was fitted with. Warnings are printed when its SOR / LPC settings differ from the current ones.

Otherwise the calibration is computed (Godbaz 4.1) and saved as `Calibrations/<SAVE_CAL_NAME>/` together with its plots:
1. LoG blob detection + GPR subpixel localisation on every calibration SL image
2. Nearest-neighbour tracking across the distances → stable dot IDs; X/Y travel mode + 3-D baseline guess from the parallax (`track_tx_space=True`: re-track in transmitter space)
3. Back-projection of every dot with the ToF cloud → U, U_tx = U − B_guess
4. Baseline B = B_guess + LS intersection of the dot lines; outlier rays (`outlier_sigma`) and dots seen at fewer than `min_points` distances are removed
5. Unit vector V_i per dot
6. 1-D dot trails from `z_min` to ∞, σ_u (trail residual), σ_u_ab (runtime AB estimator) and the σ_ToF(z) table

In [2]:
if LOAD_CALIBRATION:
    cal = calibrate.load_saved_calibration(dotCal, LOAD_CAL_NAME, camera=CAMERA,
                                           light_path_correction=LIGHT_PATH_CORRECTION)
else:
    cal = calibrate.calibrate(CAMERA, dotCal, subpixel_mode="GPR", track_tx_space=False,
                              outlier_sigma=3.0, min_points=3, z_min=0.2,
                              light_path_correction=LIGHT_PATH_CORRECTION)
    figures = calibrate.calibration_figures(dotCal, cal)
    plt.show()
    calibrate.save_calibration(cal, SAVE_CAL_NAME, figures)

Calibration distances: [0.456, 0.5, 0.556, 0.627, 0.854, 1.052, 1.378, 2.014, 3.858]


Dots per distance: [93, 93, 95, 96, 100, 100, 100, 100, 80]
Travel mode: y  (du/dw = -0.32, dv/dw = -21.75 px·m)
B_guess = [  0.65 -42.54  36.78] mm  (from 100 dots)
Tracked roster: 100 dots, per distance: [93, 93, 95, 96, 100, 100, 100, 100, 80]


Baseline from 94 dot lines: B = [  0.58 -42.27  37.81] mm  (AB along y: -4.2272 cm)


Unit vectors: 100 / 100
Trails: 100 dots, 108 samples on average (z: 0.2 m → ∞)


σ_u = 0.1903 px (2-D residual), σ_u_ab = 0.3834 px (runtime AB)
  σ_ToF(0.456 m) = 6.12 mm
  σ_ToF(0.491 m) = 15.62 mm
  σ_ToF(0.551 m) = 18.33 mm
  σ_ToF(0.625 m) = 12.71 mm
  σ_ToF(0.849 m) = 5.61 mm
  σ_ToF(1.049 m) = 3.41 mm
  σ_ToF(1.368 m) = 9.65 mm
  σ_ToF(2.017 m) = 7.55 mm
  σ_ToF(3.858 m) = 7.64 mm


Angle error V_i ↔ calibration points: median 0.06382°, max 0.69029°


/tmp/ipykernel_23509/2284451223.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Calibration saved to /home/julian/Documents/Github/iToF-SL-Combination/Calibrations/SchmersalReal_6.11_paperlike_sor6-0.40_06.11/calibration.json (+5 plots)


## SL tilt correction (projector drift)

The Coherent projector is not rigidly mounted. A small rotation after the calibration moves every dot by the same few pixels along its trail, so the SL depth is off by a nearly constant amount in 1/z (e.g. 33x33 reading ≈ 9 cm too far at 1.1 m against the 10x10 on 07.10). Only the depth labels of the trails are shifted; the saved calibration stays unchanged.

Give either what you see ("the SL reads `offset_mm` too far at `z_ref`") or the tilt directly (`tilt_mrad`, MPI_sweep_real_v11: 07.10 → ≈ +3.9). Leave all of them as `None` to switch the correction off.

In [3]:
cal_test = sl_tilt_correction.correct_cal(cal, offset_mm=None, z_ref=1.10, tilt_mrad=None)

SL tilt correction off — trails as calibrated


## Test Scene Setup

Load the test scene, its ground truth and the precomputed trails. The second figure shows what SOR removed from the test cloud.

In [4]:
# Flat wall 1500 mm (6.11, flat_1500mm_SL2_Pos0), ToF from RAW via itof_raw_bridge (dots mode)
test_dir = Path("/home/julian/Documents/Github/iToF-SL-Combination/Results/flatwall_6.11/10x10/flat_1500mm_SL2_Pos0/tof")
SL_TEST = None   # SL image = grayValue of the PCD (as in the GUI RAW tests)
TOF_TEST = str(test_dir / "flat_1500mm_SL2_Pos0.pcd")
approaches.GT_DISTANCE = 1.5584   # [m] LDS + 107.3 mm; DotCalibration.parse_distance_from_name(SL_TEST) for files like SL_0.81m.exr

setup = approaches.common_setup(dotCal, cal_test, SL_TEST, TOF_TEST)

fig, ax = plt.subplots(figsize=(8, 6))
ax.imshow(setup["sl_gray"], cmap="gray")
for t in setup["trails"]:
    if t is not None:
        ax.plot(t["u"], t["v"], lw=0.6, alpha=0.7, color="cyan")
ax.set_title("Test SL image with calibration trails")
ax.axis("off")
plot_sor_effect(dotCal, TOF_TEST)
plt.show()

SL image: gray values of the ToF cloud (no EXR given)
Trails: 100 / 100 dots calibrated (mode: y)


SOR on flat_1500mm_SL2_Pos0.pcd: 307200 points, 2731 with a ToF return
  kept 2244, rejected 487 (17.8%)   |   max mean-6NN distance = 5.53 mm


/tmp/ipykernel_23509/834313711.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Approach 1: Consistency Error  (Microsoft-Paper §4.2)

Pure trail scan, no dot localisation, no runtime triangulation. For every calibrated dot trail, the ToF depth image is resampled along the trail. The sample k\* = argmin ε with **ε = |1/Z_ToF − 1/Z_tri|** (Z_tri precomputed per sample) is selected and the **ToF depth** at k\* is returned. Dots with ε\* above `approaches.EPS_THRESHOLD` are invalidated. The standard deviation σ (most meaningful on planar scenes) is shown in the figure legends.

In [5]:
approaches.EPS_THRESHOLD = 0.75   # [1/m] ≈ 4 cm at 0.35 m, 0.3 m at 1 m
res1 = approaches.run_approach_1(dotCal, cal_test, setup, make_plots=True, dot_sample=[29, 60])
approaches.print_average_error(1, res1["depths"])
plt.show()

Approach 1: scanned 100 trails (5 invalidated by ε-threshold 0.75)
  ε range: 1.695e-04 – 1.467e-01
  Z_out (ToF @ min-ε): n=95/100  mean=1.5629 m  σ=7.50 mm



  ** Approach 1 — Average Error **
     GT distance:  1.5584 m
     Valid dots:   95 / 100
     Mean error:   +0.0045 m
     Mean |error|: 0.0065 m
     Std error:    0.0075 m
     Min error:    -0.0191 m
     Max error:    +0.0220 m
AVG_ERR|1|1.5584|95|100|0.004517|0.006531|0.007495|-0.019107|0.022041


/tmp/ipykernel_23509/40890751.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Approach 2: Gaussian Maximum-Likelihood Fusion  (Agresti & Zanuttigh)

Dot identification is the same as in Approach 3 (AB peaks along the trail → peak selection → quadratic start point → 2-D dot centre projected onto the trail → triangulation). Each modality is then modelled as **one Gaussian**. We use no SL phase information and have exactly one pixel position per dot, so the 7x7 weighted Gaussian is not needed.

- σ_SL = z²/(‖B‖·f)·σ_u  (calibrated σ_u → Agresti's d⁴/B² variance scaling)
- σ_ToF(z) interpolated from the calibration flat-wall noise table

The joint likelihood is the product of the two Gaussians. The depth at its maximum (the precision-weighted mean) is returned. σ_u is floored at `approaches.SIGMA_U_FLOOR`. Dots where SL and iToF disagree by more than `FUSION_MAX_SIGMA` σ are not fused and are counted as conflicts. Standard deviations for **SL, iToF and the joint estimate** are printed.

In [6]:
res2 = approaches.run_approach_2(dotCal, cal_test, setup, make_plots=True)
approaches.print_average_error(2, res2["depths"])
plt.show()

Approach 2: 100 trails, 91 fused depths (0 SL/iToF conflicts > 20σ not fused)
  SL (triangulated): n=91/100  mean=1.5752 m  σ=46.75 mm
  iToF: n=91/100  mean=1.5623 m  σ=7.52 mm
  Joint likelihood: n=91/100  mean=1.5689 m  σ=17.64 mm
  mean predicted σ_SL: 43.22 mm
  mean predicted σ_ToF: 36.07 mm
  mean predicted σ_joint: 27.46 mm



  ** Approach 2 — Average Error **
     GT distance:  1.5584 m
     Valid dots:   91 / 100
     Mean error:   +0.0105 m
     Mean |error|: 0.0170 m
     Std error:    0.0176 m
     Min error:    -0.0332 m
     Max error:    +0.0612 m
AVG_ERR|2|1.5584|91|100|0.010482|0.016995|0.017636|-0.033158|0.061155


/tmp/ipykernel_23509/1089758072.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Approach 3: Active Brightness Trail  (Microsoft-Paper §4.2)

1. Resample the active brightness along each dot trail (1-D vector)
2. Find peaks against a local baseline (contrast + noise prominence; trails intersect, so multiple peaks are expected)
3. Select the peak: per trail the one whose residual r = 1/Z_ToF − 1/Z_tri is closest to the **local iToF bias** b. b is voted by the `BIAS_NEIGHBOURS` nearest trails, each with its peaks that sit on a real 2-D dot (`BLOB_*`) and have a calm iToF (robust 5×5 spread < `TOF_MAD_MAX`). On a flat wall b ≈ 0 and this is the min-ε peak; on MPI_Test_2 the iToF reads ≈ 7 cm long on the face (b ≈ −1 1/m, half a dot pitch), and the selection follows it. Admissible are peaks whose iToF is at most `MPI_MAX_REL` longer than the SL depth and at most `MPI_TOL_W` (1/z) shorter; the selected (and refined) peak must lie within −`BIAS_GATE_W` … +`BIAS_GATE_SHORT_W` of b. Equal support → the smaller bias wins (`BIAS_PRIOR`), since a shift by one whole dot pitch is also self-consistent on a regular grid.
4. Subpixel: a **locally fitted quadratic** on the 1-D AB signal gives the start point. Then the **2-D dot centre** in a patch around it is projected onto the trail (`approaches.AB_SUBPIXEL_MODE`: `"fast2d"` = fast stand-in for the calibration's GPR estimator, `"GPR"` = the GPR itself (slow), `"quad1d"` = quadratic only)
5. Return the **triangulated** depth at the refined position

In [7]:
res3 = approaches.run_approach_3(dotCal, cal_test, setup, make_plots=True)
approaches.print_average_error(3, res3["depths"])
plt.show()

Approach 3: 100 trails (0 without AB peaks, 9 without usable iToF, 0 too far from the local iToF bias; 9 invalid in total)
  local iToF bias 1/z_ToF − 1/z_SL: median +0.000 1/m (5–95 %: -0.050 … +0.020)
  Z_out (triangulated): n=91/100  mean=1.5752 m  σ=46.75 mm


/home/julian/Documents/Github/iToF-SL-Combination/codePaperlike/approaches.py:698: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, ax = plt.subplots(figsize=(10, 7))



  ** Approach 3 — Average Error **
     GT distance:  1.5584 m
     Valid dots:   91 / 100
     Mean error:   +0.0168 m
     Mean |error|: 0.0392 m
     Std error:    0.0468 m
     Min error:    -0.1357 m
     Max error:    +0.1323 m
AVG_ERR|3|1.5584|91|100|0.016758|0.039205|0.046751|-0.135715|0.132319


/tmp/ipykernel_23509/505242382.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3-D reconstruction → PCD

Turns the dot depths back into 3-D points (pinhole back-projection with `K`, axial Z) and writes binary PCDs in mm, like the source clouds, so they overlay directly on the ToF cloud in CloudCompare.

- `mode="dot"`: every dot becomes a disc of `dot_size_px` pixels diameter
- `mode="area"`: a surface between the dots. 1/Z is interpolated linearly over a Delaunay mesh of the dot pixels, so planes stay planar. Triangles with an edge longer than `area_max_edge_px` stay empty.
- Pass the approaches to export as a dict; with several and `combined=True`, an extra combined file is written (A1 blue, A2 green, A3 red)

Fields: `x y z rgb dot approach` (`dot` = calibrated dot id, −1 for interpolated points).

In [8]:
clouds = reconstruction.export_pcds(
    {3: res3}, cal_test["K"], setup["H"], setup["W"],
    out_dir=Path.cwd().parent / "Results" / "pcd_export" / "flatwall_6.11_10x10",
    stem=f"{Path(TOF_TEST).parent.name}_{Path(TOF_TEST).stem}",
    mode="area", dot_size_px=3, area_step_px=1, area_max_edge_px=50, combined=False)
reconstruction.plot_reconstruction(clouds)
plt.show()

Approach 3: 91 dots → 91 points   /home/julian/Documents/Github/iToF-SL-Combination/Results/pcd_export/flatwall_6.11_10x10/tof_flat_1500mm_SL2_Pos0_A3_area1px.pcd


/tmp/ipykernel_23509/1922509700.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
